In [1]:
!pip install opensmile librosa torchaudio transformers scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.0/996.0 kB 25.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 70.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import os
import torch
import torchaudio
import numpy as np
from tqdm import tqdm
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Feature extraction with Wav2Vec2.0
def extract_wav2vec_features(dataset_path):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
    model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device)

    features = []
    labels = []

    for label, folder in enumerate(['real', 'fake']):
        folder_path = os.path.join(dataset_path, folder)
        files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]
        print(f"\nExtracting features from '{folder}' files...")

        for file in tqdm(files, desc=f"Processing {folder}", unit="file"):
            file_path = os.path.join(folder_path, file)
            audio, sr = torchaudio.load(file_path)
            
            # Resample if necessary (Wav2Vec2 expects 16kHz)
            if sr != 16000:
                resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                audio = resampler(audio)

            inputs = processor(audio.squeeze(), sampling_rate=16000, return_tensors="pt", padding=True)
            with torch.no_grad():
                outputs = model(inputs.input_values.to(device))
            embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            features.append(embeddings)
            labels.append(label)

    return np.vstack(features), np.array(labels)

# Main workflow
dataset_path = '/kaggle/input/scenefake/train'  
X, y = extract_wav2vec_features(dataset_path)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
clf = SVC(kernel='rbf', random_state=42)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred):.4f}")

2025-04-16 17:54:32.123457: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744826072.363746      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744826072.436961      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

...


Extracting features from 'real' files...


Processing real: 100%|██████████| 2525/2525 [01:25<00:00, 29.59file/s]



Extracting features from 'fake' files...


Processing fake: 100%|██████████| 10660/10660 [05:51<00:00, 30.31file/s]



Accuracy: 0.8024
Precision: 0.8024
Recall: 1.0000
F1-score: 0.8904


In [2]:
import joblib

# Save the trained model
model_path = "wav2vec_svm_model.pkl"
joblib.dump(clf, model_path)
print(f"Model saved to {model_path}")

Model saved to wav2vec_svm_model.pkl


In [3]:
import os
import torch
import torchaudio
import numpy as np
from transformers import Wav2Vec2FeatureExtractor, WavLMModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

# Feature extraction with WavLM
def extract_wavlm_features(dataset_path):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base")
    model = WavLMModel.from_pretrained("microsoft/wavlm-base").to(device)
    
    features = []
    labels = []

    for label, folder in enumerate(['real', 'fake']):
        folder_path = os.path.join(dataset_path, folder)
        files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]
        print(f"\nExtracting features from '{folder}' files...")

        for file in tqdm(files, desc=f"Processing {folder}", unit="file"):
            file_path = os.path.join(folder_path, file)
            audio, sr = torchaudio.load(file_path)

            # Resample if needed
            if sr != 16000:
                resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                audio = resampler(audio)

            inputs = processor(audio.squeeze(), sampling_rate=16000, return_tensors="pt", padding=True)
            with torch.no_grad():
                outputs = model(inputs.input_values.to(device))
            embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            features.append(embeddings)
            labels.append(label)

    return np.vstack(features), np.array(labels)

# Main workflow
dataset_path = '/kaggle/input/scenefake/train'
X, y = extract_wavlm_features(dataset_path)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred):.4f}")

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.24k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]


Extracting features from 'real' files...


Processing real:   0%|          | 4/2525 [00:00<03:22, 12.47file/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Processing real: 100%|██████████| 2525/2525 [01:16<00:00, 33.21file/s]



Extracting features from 'fake' files...


Processing fake: 100%|██████████| 10660/10660 [05:18<00:00, 33.47file/s]



Accuracy: 0.9662
Precision: 0.9725
Recall: 0.9858
F1-score: 0.9791


In [4]:
# Save the trained model
model_path = "logistic_model_wavlm.pkl"
joblib.dump(clf, model_path)
print("Model saved as 'logistic_model_wavlm.pkl'")

Model saved as 'logistic_model_wavlm.pkl'
